# SIESTA Example notebook

In this notebook, I show you how to use SIESTA for two general Echelle spectrograph layouts so you can re-use it for your owns: a nighttime large-spectral range spectrograph (Instrument A) and an integral-field solar spectrograph with a shorter target spectral range (Instrument B). Both are meant for visible light. 
For each instrument, I show 2 versions: a first one with a clear flaw in its design, and a second one correcting for it. This highlights the main reason for SIESTA to exist, which is helping choices in the early design phase of an Echelle spectrograph.

**Instrument A - Nighttime spectrograph**

The instrument has the following target specifications: 
- Spectral range: 400-700 nm
- Resolving power R: >50'000


**Instrument B - Solar integral-field spectrograph**

The instrument has the following target specifications: 
- Spectral range: 600-670 nm
- Resolving power R: >150'000
- Integral field-of-view: 2"x2" subdivided in 4 1"x1" sub-images


In [1]:
from siesta_package.siesta_utils import *


# Instrument A - Nighttime spectrograph

The instrument has the following target specifications: 
- Spectral range: 400-700 nm
- Resolving power R: >50'000

General parameters

In [2]:
### Select spectral range in nm
spectral_range = (400, 700)

### Select spectral pitch of spectral points to compute (for each spatial element): This defines the spacing between two consecutive spectral points in the simulated image. By default, set it to your target sampling resolution. Here, R > 50'000 is required so the minimum sampling is 400/50000 = 0.008 nm. If you want to oversample, you can set it to a smaller value.
spectral_res = 0.008 #nm



## First iteration - flawed

Slit

In [3]:
slit = Slit(name="slit", width=0.05, height=0.2)

Collimating element

In [4]:
collimator = Lens(name="collimator", focal_length=800, diameter=50)

Echelle grating

In [5]:
echelle = EchelleGrating(name="121E", groove_density=110, blaze_angle=64, semi_deviation_angle_deg=0)

Cross-disperser: 2x60° dispersion prisms in F2

In [6]:
# F2 prism
crossdisperser1 = Prism(name="disperser prism", glass_type=["specs","SCHOTT-optical","F2"], beam_diameter=20, base=50, apex_angle_deg=60, input_angle_deg=45, manual=False, spectral_range_nm=spectral_range, spectral_res_nm=spectral_res)
crossdisperser2 = Prism(name="disperser prism 2", glass_type=["specs","SCHOTT-optical","F2"], beam_diameter=20, base=50, apex_angle_deg=60, input_angle_deg=39, manual=False, prev=crossdisperser1, spectral_range_nm=spectral_range, spectral_res_nm=spectral_res)

Sellmeier coefficients for specs SCHOTT-optical F2: [0.0, 1.34533359, 0.00997743871, 0.209073176, 0.0470450767, 0.937357162, 111.886764]


Camera lens

In [7]:
camera = Lens(name="camera", focal_length=200, diameter=50)

Imager

In [8]:
## Andor Zyla
zyla = Camera_sensor(name="Andor Zyla", pixels_x=2560, pixels_y=2160, pixel_size=6.5) #um

Assembling our version 1 instrument

In [9]:
centers_list = [(0.5*zyla.size_x_mm, 0.5*zyla.size_y_mm)]

instrumentA_v1 = Instrument(name="Instrument A v1", spectral_range=spectral_range, spatial_centers=centers_list, spectral_res_nm=spectral_res, echelle=echelle, disperser=crossdisperser2, camera_lens=camera, collimator_lens=collimator, slit=slit, camera_sensor=zyla, wavelength_scan_width_nm=1)


Sellmeier coefficients for specs SCHOTT-optical F2: [0.0, 1.34533359, 0.00997743871, 0.209073176, 0.0470450767, 0.937357162, 111.886764]
Sellmeier coefficients for specs SCHOTT-optical F2: [0.0, 1.34533359, 0.00997743871, 0.209073176, 0.0470450767, 0.937357162, 111.886764]
refractive index n for 550.0039999996504 nm: 1.5988645728658888
input angle lambda0 in rad for disperser prism : 0.7853981633974483
output angle lambda0 in rad for disperser prism : -1.09369693427946
input angle lambda0 in rad for disperser prism 2 : 0.6341790251949262
output angle lambda0 in rad for disperser prism 2 : -1.4279103511408335
-1.4279103511408335
Sellmeier coefficients for specs SCHOTT-optical F2: [0.0, 1.34533359, 0.00997743871, 0.209073176, 0.0470450767, 0.937357162, 111.886764]
input angle in rad for disperser prism 2 : 0.6329962182129234, n=1.5994264511326548, exit angle: -1.4375708198434183


Starting the SIESTA app

In [10]:
instrumentA_v1.plotCD()


Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>

Here below is the screenshot of the current instrument version. By running the simulation, one can attest that R > 50'000 over the whole 400-700 nm spectral range. However, it is clear that the spectrum is too large to fit in the selected camera sensor.

<img src="Example_InstrumentAv1.png">

Possible files to export from the simulation

In [11]:
# Exporting exact positions of Neon spectral lines present in the instrument spectral range (no scanning windows) as a fits file.
# instrumentA_v1.exportAsFits(species=["Neon"], filename="neon_instrumentA_v1", path="", spectral_range_nm=spectral_range, wantSlitKernel=True)

# Exporting exact positions of Thorium spectral lines present in the instrument spectral range (no scanning windows) as a fits file. These were selected as the 5 % most intense lines of Thorium according to the NIST database. 
# instrumentA_v1.exportAsFits(species=["Thorium"], filename="thorium_instrumentA_v1", path="", spectral_range_nm=spectral_range, wantSlitKernel=True)

## Second iteration - corrected version

The first version has a clear issue with the spectrum width. One can correct this by using a different Echelle grating that will be used at higher orders, so thinner in spectral width for the same resolving power. However the cross-dispersion system will need to disperse more to separate these higher orders. Here, I propose to a single grating cross-disperser working at its first order of diffration. The 400-700 nm spectral range is suitable for such a system. 

In [12]:
echelle = EchelleGrating(name="053E", groove_density=52.91, blaze_angle=64, semi_deviation_angle_deg=0)

In [13]:
crossdisperser_grating = Grating(name="new cross-disperser", groove_density=200, alpha=17)

In [14]:
instrumentA_v2 = Instrument(name="Instrument A v2", spectral_range=spectral_range, spatial_centers=centers_list, spectral_res_nm=spectral_res, echelle=echelle, disperser=crossdisperser_grating, camera_lens=camera, collimator_lens=collimator, slit=slit, camera_sensor=zyla, wavelength_scan_width_nm=1)
instrumentA_v2.plotCD()

-0.1833972579915723
Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>

Here below is the corrected design of Instrument A. The whole range fits inside the specified sensor and all the orders are clearly separated. By running the simulation with the previous cell, one can validate that R > 50'000 everywhere. Also, the projected slit width is 12 microns on the sensor, which is close to 2 pixels (6.5 um pixels).  

<img src="Example_InstrumentAv2.png">

# Instrument B - Solar integral-field spectrograph

The instrument has the following target specifications: 
- Spectral range: 600-670 nm
- Resolving power R: > 150'000
- Integral field-of-view: 2"x2" subdivided in 4 1"x1" sub-images

General parameters

In [15]:
### Select spectral range in nm
spectral_range = (600, 670)

### Select spectral pitch of spectral points to compute (for each spatial element): This defines the spacing between two consecutive spectral points in the simulated image. By default, set it to your target sampling resolution. Here, R > 150'000 is required so the minimum sampling is 600/150000 = 0.004 nm.
spectral_res = 0.004 #nm


## First iteration - flawed

Integral-field unit: creating 4 1"x1" sub-images from a 2"x2" intial field-of-view. I define a "unit slit" to be one sub-image, thus 1"x1". Let's assume a solar telescope with a 5"/mm plate scale leading to a physical 0.2mmx0.2mm unit slit size.

In [16]:
unit_slit = Slit(name="unit element IFU", width=0.2, height=0.2) #mm

Collimating element

In [17]:
collimator = Lens(name="collimator", focal_length=2000, diameter=50)

Echelle grating

In [18]:
echelle = EchelleGrating(name="412E", groove_density=23.2, blaze_angle=63, semi_deviation_angle_deg=0)

Cross-disperser: grating

In [19]:
crossdisperser_grating = Grating(name="new cross-disperser", groove_density=200, alpha=17)

Camera lens

In [20]:
camera = Lens(name="camera", focal_length=400, diameter=50)

Imager

In [21]:
## Andor Zyla
zyla = Camera_sensor(name="Andor Zyla", pixels_x=2560, pixels_y=2160, pixel_size=6.5) #um

Assembling the version 1 instrument

In [22]:
shift_y = 0.075 #mm shift in y from initial position. For a long-slit reformatting IFU, this corresponds to half the height of the projected unit slit on the camera sensor.

centers_list = [(zyla.size_x_mm/2, 3/5*zyla.size_y_mm - 2*shift_y),
                (zyla.size_x_mm/2, 3/5*zyla.size_y_mm -  shift_y),
                (zyla.size_x_mm/2, 3/5*zyla.size_y_mm),
                (zyla.size_x_mm/2, 3/5*zyla.size_y_mm +  shift_y)]

instrumentB_v1 = Instrument(name="Instrument B v1", spectral_range=spectral_range, spatial_centers=centers_list, spectral_res_nm=spectral_res, echelle=echelle, disperser=crossdisperser_grating, camera_lens=camera, collimator_lens=collimator, slit=unit_slit, camera_sensor=zyla, wavelength_scan_width_nm=1)


-0.16613448788511417
-0.16613448788511417
-0.16613448788511417
-0.16613448788511417


Starting the SIESTA app

In [23]:
instrumentB_v1.plotCD()


Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>

Here below is the version 1 of the Instrument B. I show that the H-alpha and the Fe I 6301 and 6302 lines are correctly imaged inside the sensor. The 600-670 nm spectral range was roughly chosen to capture those. However, the resolving power is greatly inferior to the initial target of R > 150'000. To reach the target resolving power, stronger dispersion is needed. 

<img src="Example_InstrumentBv1.png">

## Second iteration - corrected version

The first version has a clear issue with the resolving power. To increase it, one can choose to illuminate a higher number of Echelle grooves, therefore having a bigger Echelle grating. To fully illuminate it, one then needs a longer focal length for the collimator. Finally, the camera lens system usually has to use longer focal lengths too.

In the end, the complete target spectrum is larger and thus a choice has to be made: a larger imager for capturing the same spectral range, or keeping the same camera sensor and reduce the spectral range e.g. only capture a spectral line ? Here, I show the first choice by also increasing the size of the camera sensor.

In [24]:
collimator_v2 = Lens(name="collimator", focal_length=5000, diameter=50)

In [25]:
echelle_v2 = EchelleGrating(name="412E", groove_density=23.2, blaze_angle=63, semi_deviation_angle_deg=5)

In [26]:
camera_v2 = Lens(name="camera", focal_length=1600, diameter=50)

In [27]:
## Andor Balor
balor = Camera_sensor(name="Andor Balor", pixels_x=4128, pixels_y=4104, pixel_size=12) #um
shift_y = 0.064 #mm shift in y from initial position. For a long-slit reformatting IFU, this corresponds to half the height of the projected unit slit on the camera sensor.

centers_list = [(balor.size_x_mm/2, 3/5*balor.size_y_mm - 2*shift_y),
                (balor.size_x_mm/2, 3/5*balor.size_y_mm -  shift_y),
                (balor.size_x_mm/2, 3/5*balor.size_y_mm),
                (balor.size_x_mm/2, 3/5*balor.size_y_mm +  shift_y)]


In [28]:
instrumentB_v2 = Instrument(name="Instrument B v2", spectral_range=spectral_range, spatial_centers=centers_list, spectral_res_nm=spectral_res, echelle=echelle_v2, disperser=crossdisperser_grating, camera_lens=camera_v2, collimator_lens=collimator_v2, slit=unit_slit, camera_sensor=balor, wavelength_scan_width_nm=1)
instrumentB_v2.plotCD()

-0.16613448788511417
-0.16613448788511417
-0.16613448788511417
-0.16613448788511417
Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>

With the new configuration, all the initial requirements are met. However, the Echelle grating is also off-Littrow to reach the target R, and brings additional challenges to get a good image quality. SIESTA allows the user to observe a part of the effects of many tweaks but not all of them. SIESTA will only get you an idea of what would be the general layout of the target instrument.  

<img src="Example_InstrumentBv2.png">